## ChromaDB

In [1]:
# Installation
# !uv add chromadb

In [2]:
import chromadb

1. Création du client ChromaDB

In [3]:
client = chromadb.Client()

# Client persistant (Pour la production)
# client = chromadb.PersistentClient(path='./chroma_db')

2. Configuration de l'embedding function

In [4]:
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

c:\Users\Administrateur\Documents\M2i_CDSD_TDTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13085.83it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Création d'une collection

In [5]:
collection = client.get_or_create_collection(
    name="documentation",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

4. Ajout des documents

In [6]:
documents = [
    "Python est un langage de programmation interprété et polyvalent.",
    "Le machine learning permet aux ordinateurs d'apprendre à partir de données.",
    "Les réseaux de neurones sont inspirés du fonctionnement du cerveau humain.",
    "Docker permet de containeriser des applications pour un déploiement facile.",
    "FastAPI est un framework web moderne et performant pour Python.",
    "Les bases de données vectorielles stockent des embeddings pour la recherche sémantique.",
    "RAG combine recherche documentaire et génération de texte par LLM.",
    "Git est un système de contrôle de version distribué.",
    "Kubernetes orchestre les conteneurs à grande échelle.",
    "L'API REST utilise les méthodes HTTP pour manipuler des ressources.",
]


metadatas = [
    {"category": "programming", "difficulty": "beginner"},
    {"category": "ml", "difficulty": "intermediate"},
    {"category": "ml", "difficulty": "advanced"},
    {"category": "devops", "difficulty": "intermediate"},
    {"category": "programming", "difficulty": "intermediate"},
    {"category": "ml", "difficulty": "intermediate"},
    {"category": "ml", "difficulty": "advanced"},
    {"category": "devops", "difficulty": "beginner"},
    {"category": "devops", "difficulty": "advanced"},
    {"category": "programming", "difficulty": "beginner"},
]

ids = [f"doc_{i}" for i in range(len(documents))]

In [7]:
collection.add(documents=documents, metadatas=metadatas, ids=ids)

5. Recherche

In [8]:
query = "Comment fonctionne l'intelligence artificielle ?"

results = collection.query(
    query_texts=[query], n_results=3, include=["documents", "distances", "metadatas"]
)

for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    distance = results["distances"][0][i]
    metadata = results["metadatas"][0][i]

    print(f"similarité : {distance}")
    print(f"Catégorie : {metadata['category']}")
    print(f"Document : {doc}")

similarité : 0.5854740142822266
Catégorie : ml
Document : Le machine learning permet aux ordinateurs d'apprendre à partir de données.
similarité : 0.6572716236114502
Catégorie : ml
Document : Les réseaux de neurones sont inspirés du fonctionnement du cerveau humain.
similarité : 0.8222171664237976
Catégorie : devops
Document : Docker permet de containeriser des applications pour un déploiement facile.


6. Recherche avec filtres

In [9]:
results_filtered = collection.query(
    query_texts=["déploiement d'applications"],
    n_results=3,
    where={"category": "devops"},
    include=["documents", "distances", "metadatas"],
)

for i in range(len(results_filtered["documents"][0])):
    doc = results_filtered["documents"][0][i]
    distance = results_filtered["distances"][0][i]
    metadata = results_filtered["metadatas"][0][i]

    print(f"similarité : {distance}")
    print(f"Catégorie : {metadata['category']}")
    print(f"Document : {doc}")


similarité : 0.5465202331542969
Catégorie : devops
Document : Docker permet de containeriser des applications pour un déploiement facile.
similarité : 0.6323938369750977
Catégorie : devops
Document : Git est un système de contrôle de version distribué.
similarité : 0.82845139503479
Catégorie : devops
Document : Kubernetes orchestre les conteneurs à grande échelle.


In [10]:
# filtre avancés
results_filtered = collection.query(
    query_texts=["déploiement d'applications"],
    n_results=3,
    where={"$and": [{"category": "devops"}, {"difficulty": "beginner"}]},
    include=["documents", "distances", "metadatas"],
)

for i in range(len(results_filtered["documents"][0])):
    doc = results_filtered["documents"][0][i]
    distance = results_filtered["distances"][0][i]
    metadata = results_filtered["metadatas"][0][i]

    print(f"similarité : {distance}")
    print(f"Catégorie : {metadata['category']}")
    print(f"Document : {doc}")


similarité : 0.6323938369750977
Catégorie : devops
Document : Git est un système de contrôle de version distribué.


7. Opérations CRUD

In [11]:
# Récupérer un document par Id
doc = collection.get(ids=["doc_0"])
print(doc)

{'ids': ['doc_0'], 'embeddings': None, 'documents': ['Python est un langage de programmation interprété et polyvalent.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'programming', 'difficulty': 'beginner'}]}


In [12]:
# Mettre à jour
collection.update(
    ids=["doc_0"],
    documents=["Python est un langage de programmation."],
    metadatas=[{"category": "dev", "difficulty": "beginner", "updated": True}],
)